In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
silver_tx_dir = PROJECT_ROOT / "data" / "silver" / "transactions"

# Read all 731 daily partitions into one table (exploration only).
tx = pd.read_parquet(silver_tx_dir)

print("rows:", len(tx))
print("columns:", tx.shape[1])
print("fraud:", int(tx["is_fraud"].sum()) if "is_fraud" in tx.columns else "no is_fraud column")
print("memory MB:", round(tx.memory_usage(deep=True).sum() / 1024**2))
print(tx.dtypes)

rows: 1852394
columns: 23
fraud: 9651
memory MB: 660
trans_num                        str
trans_ts              datetime64[us]
card_hash                        str
merchant                         str
category                         str
amt                          float64
gender                           str
city                             str
state                            str
zip                              str
lat                          float64
long                         float64
city_pop                       int64
job                              str
birth_year                     int16
merch_lat                    float64
merch_long                   float64
is_fraud                        int8
_ingested_at     datetime64[us, UTC]
_source                          str
_source_file                     str
_batch_date     date32[day][pyarrow]
batch_date                  category
dtype: object


In [2]:
def dependency_report(key, cols):
    """For each column: how many different values does one key have?"""
    per_key = tx.groupby(key, observed=True)[cols].nunique()
    return pd.DataFrame(
        {
            "max_distinct_per_key": per_key.max(),
            "keys_with_more_than_1": (per_key > 1).sum(),
        }
    )


print("=== distinct counts ===")
print(tx[["card_hash", "merchant", "category", "zip", "city", "state", "job"]].nunique())

print("\n=== per card ===")
print(dependency_report(
    "card_hash",
    ["gender", "birth_year", "job", "city", "state", "zip", "lat", "long", "city_pop"],
))

print("\n=== per merchant ===")
print(dependency_report("merchant", ["category", "merch_lat", "merch_long"]))

print("\n=== per zip ===")
print(dependency_report("zip", ["city", "state", "lat", "long", "city_pop", "card_hash"]))

print("\nmerchant names starting with 'fraud_':",
      round(float(tx["merchant"].str.startswith("fraud_").mean()), 4))

=== distinct counts ===
card_hash    999
merchant     693
category      14
zip          985
city         906
state         51
job          497
dtype: int64

=== per card ===
            max_distinct_per_key  keys_with_more_than_1
gender                         1                      0
birth_year                     1                      0
job                            1                      0
city                           1                      0
state                          1                      0
zip                            1                      0
lat                            1                      0
long                           1                      0
city_pop                       1                      0

=== per merchant ===
            max_distinct_per_key  keys_with_more_than_1
category                       2                      7
merch_lat                   6262                    693
merch_long                  6261                    693

=== per zip ===
   

In [3]:
cats_per_merchant = tx.groupby("merchant", observed=True)["category"].nunique()
multi = cats_per_merchant[cats_per_merchant > 1].index

detail = (
    tx[tx["merchant"].isin(multi)]
    .groupby(["merchant", "category"], observed=True)
    .agg(
        rows=("trans_num", "size"),
        fraud=("is_fraud", "sum"),
        first_ts=("trans_ts", "min"),
        last_ts=("trans_ts", "max"),
    )
    .reset_index()
    .sort_values(["merchant", "category"])
)
print(detail.to_string(index=False))

pairs = tx[["merchant", "category"]].drop_duplicates().shape[0]
print("\nmerchant-category pairs:", pairs)

      merchant       category  rows  fraud            first_ts             last_ts
     Boyer PLC       misc_pos  2282      4 2019-01-01 01:46:40 2020-12-31 15:42:59
     Boyer PLC   shopping_net  2717     51 2019-01-02 14:41:09 2020-12-31 21:59:20
   Cormier LLC health_fitness  2429      3 2019-01-01 16:46:45 2020-12-31 21:06:35
   Cormier LLC   shopping_net  2817     50 2019-01-01 14:31:02 2020-12-31 09:10:00
 Dickinson Ltd       misc_pos  2269      4 2019-01-01 03:10:23 2020-12-31 19:12:05
 Dickinson Ltd  personal_care  2684      5 2019-01-01 16:51:12 2020-12-31 23:44:29
Gislason Group       misc_pos  2392     10 2019-01-01 05:36:54 2020-12-31 16:10:40
Gislason Group         travel  1164      3 2019-01-01 15:39:20 2020-12-31 17:45:42
   Kilback LLC    food_dining  2646      9 2019-01-01 15:25:28 2020-12-31 23:33:17
   Kilback LLC    grocery_pos  3616     53 2019-01-01 02:17:46 2020-12-31 11:07:24
      Kuhn LLC       misc_net  1736     27 2019-01-01 01:22:14 2020-12-31 11:19:33
    

In [4]:
merchant_dim = tx[["merchant", "category"]].drop_duplicates()
print(merchant_dim.groupby("category").size().sort_values())
print("\ntotal:", len(merchant_dim))

category
entertainment     50
food_dining       50
gas_transport     50
grocery_net       50
grocery_pos       50
health_fitness    50
home              50
kids_pets         50
misc_net          50
misc_pos          50
personal_care     50
shopping_net      50
shopping_pos      50
travel            50
dtype: int64

total: 700


In [5]:
silver_dir = PROJECT_ROOT / "data" / "silver"
acs = pd.read_parquet(silver_dir / "census_acs")
zcta = pd.read_parquet(silver_dir / "census_zcta")

tx_zips = pd.DataFrame({"zip": tx["zip"].unique()})
print("transaction zips:", len(tx_zips))
print("acs rows:", len(acs), "| zcta rows:", len(zcta))

check = tx_zips.merge(
    zcta[["zip", "land_area_m2"]], on="zip", how="left", indicator="in_zcta"
).merge(
    acs[["zip", "population", "median_household_income", "median_age"]],
    on="zip", how="left", indicator="in_acs",
)

print("\nmissing in zcta:", int((check["in_zcta"] == "left_only").sum()))
print("missing in acs:", int((check["in_acs"] == "left_only").sum()))
print("null income:", int(check["median_household_income"].isna().sum()))
print("null age:", int(check["median_age"].isna().sum()))
print("zero population:", int((check["population"] == 0).sum()))
print("censored income:", int(check["median_household_income"].isin([2499, 250001]).sum()))

no_income_zips = check.loc[check["median_household_income"].isna(), "zip"]
affected = tx["zip"].isin(no_income_zips)
print("\ntransactions without income:", int(affected.sum()))
print("fraud among them:", int(tx.loc[affected, "is_fraud"].sum()))

transaction zips: 985
acs rows: 33120 | zcta rows: 33144

missing in zcta: 0
missing in acs: 0
null income: 57
null age: 12
zero population: 8
censored income: 0

transactions without income: 124334
fraud among them: 539


In [6]:
no_income = check[check["median_household_income"].isna()]
has_income = check[check["median_household_income"].notna()]

print("=== population of ZIPs WITHOUT income ===")
print(no_income["population"].describe().round(0))
print("under 100 people:", int((no_income["population"] < 100).sum()), "of", len(no_income))

print("\n=== population of ZIPs WITH income ===")
print(has_income["population"].describe().round(0))
print("under 100 people:", int((has_income["population"] < 100).sum()), "of", len(has_income))

=== population of ZIPs WITHOUT income ===
count      57.0
mean      570.0
std      1215.0
min         0.0
25%        42.0
50%       147.0
75%       390.0
max      6198.0
Name: population, dtype: float64
under 100 people: 22 of 57

=== population of ZIPs WITH income ===
count       928.0
mean      10250.0
std       14596.0
min          34.0
25%         942.0
50%        3333.0
75%       14540.0
max      105304.0
Name: population, dtype: float64
under 100 people: 15 of 928


In [7]:
hol = pd.read_parquet(silver_dir / "holidays")

print("rows:", len(hol))
print(hol.dtypes)
print()
print(hol[["holiday_date", "name", "state_code", "is_global", "types"]].head(8).to_string(index=False))

print("\nholiday state_code examples:", list(hol["state_code"].dropna().unique()[:8]))
print("transaction state examples:", list(tx["state"].unique()[:8]))

nationwide = hol[hol["state_code"].isna()]
by_state = hol[hol["state_code"].notna()]
print("\nnationwide rows:", len(nationwide), "| distinct dates:", nationwide["holiday_date"].nunique())
print("state rows:", len(by_state))
print("max rows per (date, state):", int(by_state.groupby(["holiday_date", "state_code"]).size().max()))
print("\ntypes:")
print(hol["types"].value_counts())

rows: 160
holiday_date    date32[day][pyarrow]
name                             str
country_code                     str
state_code                       str
is_global                       bool
types                            str
_ingested_at     datetime64[us, UTC]
_source                          str
_year                          int16
country                     category
year                        category
dtype: object

holiday_date                        name state_code  is_global       types
  2019-01-01              New Year's Day        NaN       True Bank,Public
  2019-01-21 Martin Luther King, Jr. Day        NaN       True Bank,Public
  2019-02-12          Lincoln's Birthday         CA      False  Observance
  2019-02-12          Lincoln's Birthday         CT      False  Observance
  2019-02-12          Lincoln's Birthday         IL      False  Observance
  2019-02-12          Lincoln's Birthday         IN      False  Observance
  2019-02-12          Lincoln's Birthday   

In [8]:
hol_flags = hol.assign(
    day=hol["holiday_date"].astype(str),
    is_public=hol["types"].str.contains("Public"),
    is_bank=hol["types"].str.contains("Bank"),
)

# Nationwide: one row per day.
nat = (
    hol_flags[hol_flags["state_code"].isna()]
    .groupby("day")[["is_public", "is_bank"]].max()
    .rename(columns={"is_public": "nat_public", "is_bank": "nat_bank"})
    .reset_index()
)

# State level: squeeze to one row per (day, state).
st = (
    hol_flags[hol_flags["state_code"].notna()]
    .groupby(["day", "state_code"])[["is_public"]].max()
    .rename(columns={"is_public": "state_public"})
    .reset_index()
    .rename(columns={"state_code": "state"})
)

base = pd.DataFrame({
    "day": tx["trans_ts"].dt.strftime("%Y-%m-%d"),
    "state": tx["state"],
    "is_fraud": tx["is_fraud"],
})
before = len(base)

# The trap: joining straight to the raw state rows.
raw_state = hol_flags.loc[hol_flags["state_code"].notna(), ["day", "state_code"]]
naive = base.merge(raw_state.rename(columns={"state_code": "state"}), on=["day", "state"], how="left")
print("naive join rows:", len(naive), "| extra rows:", len(naive) - before)

# The safe way.
t = base.merge(nat, on="day", how="left").merge(st, on=["day", "state"], how="left")
print("safe join rows:", len(t), "| extra rows:", len(t) - before)

for c in ["nat_public", "nat_bank", "state_public"]:
    t[c] = t[c].eq(True)
t["public_holiday"] = t["nat_public"] | t["state_public"]

print("\ntransactions per flag:")
print(t[["nat_public", "nat_bank", "state_public", "public_holiday"]].sum())

summary = t.groupby("public_holiday")["is_fraud"].agg(rows="size", fraud="sum")
summary["fraud_rate_pct"] = (summary["fraud"] / summary["rows"] * 100).round(3)
print()
print(summary)

naive join rows: 1853342 | extra rows: 948
safe join rows: 1852394 | extra rows: 0

transactions per flag:
nat_public        46917
nat_bank          53247
state_public       5579
public_holiday    52496
dtype: int64

                   rows  fraud  fraud_rate_pct
public_holiday                                
False           1799898   9404           0.522
True              52496    247           0.471


In [9]:
mcc = pd.read_parquet(PROJECT_ROOT / "data" / "bronze" / "mcc")

print("rows:", len(mcc))
print(mcc.dtypes)
print()
print(mcc.head(10).to_string(index=False))

rows: 981
mcc                                     str
edited_description                      str
combined_description                    str
usda_description                        str
irs_description                         str
irs_reportable                          str
_ingested_at            datetime64[us, UTC]
_source                                 str
_source_url                             str
_content_sha256                         str
version                            category
dtype: object

 mcc                                                                                                                             edited_description                                                                                                                           combined_description                                      usda_description                 irs_description irs_reportable                     _ingested_at                   _source                                         

In [10]:
candidates = {
    "entertainment":  ["7832", "7922", "7999"],
    "food_dining":    ["5812", "5814"],
    "gas_transport":  ["5541", "5542"],
    "grocery_pos":    ["5411", "5499"],
    "grocery_net":    ["5411", "5499"],
    "health_fitness": ["7997", "7941", "8099"],
    "home":           ["5712", "5200", "5719"],
    "kids_pets":      ["5995", "5641", "5945"],
    "misc_pos":       ["5999"],
    "misc_net":       ["5999", "5969"],
    "personal_care":  ["7230", "7298"],
    "shopping_pos":   ["5311", "5651"],
    "shopping_net":   ["5311", "5964", "5969"],
    "travel":         ["4722", "4511", "7011"],
}

codes = mcc["mcc"].str.strip()
print("mcc codes unique:", codes.is_unique)
lookup = dict(zip(codes, mcc["edited_description"]))

for category, options in candidates.items():
    print(f"\n{category}")
    for code in options:
        print("   ", code, "-", lookup.get(code, "NOT FOUND"))

mcc codes unique: True

entertainment
    7832 - Motion Picture Theaters
    7922 - Theatrical Producers (Except Motion Pictures), Ticket Agencies
    7999 - Recreation Services (Not Elsewhere Classified)

food_dining
    5812 - Eating places and Restaurants
    5814 - Fast Food Restaurants

gas_transport
    5541 - Service Stations ( with or without ancillary services)
    5542 - Automated Fuel Dispensers

grocery_pos
    5411 - Grocery Stores, Supermarkets
    5499 - Misc. Food Stores – Convenience Stores and Specialty Markets

grocery_net
    5411 - Grocery Stores, Supermarkets
    5499 - Misc. Food Stores – Convenience Stores and Specialty Markets

health_fitness
    7997 - Membership Clubs (Sports, Recreation, Athletic), Country Clubs, and Private Golf Courses
    7941 - Commercial Sports, Athletic Fields, Professional Sport Clubs, and Sport Promoters
    8099 - Medical Services and Health Practitioners (Not Elsewhere Classified)

home
    5712 - Furniture, Home Furnishings, and E

In [11]:
mapping = pd.DataFrame(
    [
        ("entertainment",  "7999", "unknown"),
        ("food_dining",    "5812", "unknown"),
        ("gas_transport",  "5541", "unknown"),
        ("grocery_pos",    "5411", "in_person"),
        ("grocery_net",    "5411", "online"),
        ("health_fitness", "7997", "unknown"),
        ("home",           "5712", "unknown"),
        ("kids_pets",      "5995", "unknown"),
        ("misc_pos",       "5999", "in_person"),
        ("misc_net",       "5999", "online"),
        ("personal_care",  "7230", "unknown"),
        ("shopping_pos",   "5311", "in_person"),
        ("shopping_net",   "5311", "online"),
        ("travel",         "4722", "unknown"),
    ],
    columns=["category", "mcc", "channel"],
)

tx_categories = set(tx["category"].unique())
print("rows:", len(mapping))
print("categories unique:", mapping["category"].is_unique)
print("missing from mapping:", sorted(tx_categories - set(mapping["category"])))
print("not in transactions:", sorted(set(mapping["category"]) - tx_categories))
print("codes not in MCC table:", sorted(set(mapping["mcc"]) - set(codes)))
print("distinct codes:", mapping["mcc"].nunique())

stats = (
    tx.groupby("category", observed=True)["is_fraud"]
    .agg(rows="size", fraud="sum")
    .reset_index()
)
out = mapping.merge(stats, on="category", how="left")
out["fraud_rate_pct"] = (out["fraud"] / out["rows"] * 100).round(3)
print()
print(out.to_string(index=False))
print("\ntotal rows:", int(out["rows"].sum()), "| total fraud:", int(out["fraud"].sum()))

rows: 14
categories unique: True
missing from mapping: []
not in transactions: []
codes not in MCC table: []
distinct codes: 11

      category  mcc   channel   rows  fraud  fraud_rate_pct
 entertainment 7999   unknown 134118    292           0.218
   food_dining 5812   unknown 130729    205           0.157
 gas_transport 5541   unknown 188029    772           0.411
   grocery_pos 5411 in_person 176191   2228           1.265
   grocery_net 5411    online  64878    175           0.270
health_fitness 7997   unknown 122553    185           0.151
          home 5712   unknown 175460    265           0.151
     kids_pets 5995   unknown 161727    304           0.188
      misc_pos 5999 in_person 114229    322           0.282
      misc_net 5999    online  90654   1182           1.304
 personal_care 7230   unknown 130085    290           0.223
  shopping_pos 5311 in_person 166463   1056           0.634
  shopping_net 5311    online 139322   2219           1.593
        travel 4722   unknown  

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
mcc = pd.read_parquet(PROJECT_ROOT / "data" / "bronze" / "mcc")

print("columns:", list(mcc.columns))
print("rows:", len(mcc))

biz = ["mcc", "edited_description", "combined_description",
       "usda_description", "irs_description", "irs_reportable"]
for c in biz:
    s = mcc[c]
    stripped = s.str.strip()
    print(
        c,
        "| null:", int(s.isna().sum()),
        "| empty:", int(stripped.eq("").sum()),
        "| needs trim:", int((s.notna() & (s != stripped)).sum()),
        "| distinct:", s.nunique(),
    )

print("\nmcc not 4 digits:", int((~mcc["mcc"].str.strip().str.fullmatch(r"\d{4}")).sum()))
print("mcc duplicated:", int(mcc["mcc"].duplicated().sum()))
print("\nirs_reportable values:")
print(mcc["irs_reportable"].value_counts(dropna=False))

other = [c for c in mcc.columns if c not in biz]
print("\nmetadata sample:")
print(mcc[other].head(1).T)

columns: ['mcc', 'edited_description', 'combined_description', 'usda_description', 'irs_description', 'irs_reportable', '_ingested_at', '_source', '_source_url', '_content_sha256', 'version']
rows: 981
mcc | null: 0 | empty: 0 | needs trim: 0 | distinct: 981
edited_description | null: 0 | empty: 0 | needs trim: 1 | distinct: 722
combined_description | null: 0 | empty: 8 | needs trim: 1 | distinct: 715
usda_description | null: 0 | empty: 275 | needs trim: 0 | distinct: 706
irs_description | null: 0 | empty: 18 | needs trim: 17 | distinct: 286
irs_reportable | null: 0 | empty: 18 | needs trim: 0 | distinct: 6

mcc not 4 digits: 0
mcc duplicated: 0

irs_reportable values:
irs_reportable
Yes                 810
No1.6041-3(c)       139
                     18
No1.6041-3(p)(2)      8
No1.6041-3(p)(4)      4
No1.6041-3(p)(3)      2
Name: count, dtype: int64

metadata sample:
                                                                 0
_ingested_at                      2026-09-25 21:30:5

In [2]:
import importlib
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.silver.mcc as silver_mcc
importlib.reload(silver_mcc)

files = sorted((PROJECT_ROOT / "data" / "bronze" / "mcc").rglob("*.parquet"))
print("files:", [str(f.relative_to(PROJECT_ROOT)) for f in files])

bronze_file = files[0]
version = bronze_file.parent.name.split("=", 1)[1]
mcc_bronze = pd.read_parquet(bronze_file)

mcc_silver = silver_mcc.transform_mcc(mcc_bronze, version)

print("bronze shape:", mcc_bronze.shape)
print("silver shape:", mcc_silver.shape)
print(mcc_silver.dtypes)
print()
print(mcc_silver.head(5).to_string(index=False))

desc = mcc_silver["description"]
print("\nuntrimmed left:", int((desc != desc.str.strip()).sum()))
wanted = ["4722", "5311", "5411", "5541", "5712", "5812",
          "5995", "5999", "7230", "7997", "7999"]
print("mapping codes found:", int(mcc_silver["mcc"].isin(wanted).sum()), "of", len(wanted))

files: ['data\\bronze\\mcc\\version=9675cfab77d6\\part-0.parquet']
bronze shape: (981, 10)
silver shape: (981, 6)
mcc                                str
description                        str
_ingested_at       datetime64[us, UTC]
_source                            str
_content_sha256                    str
_version                           str
dtype: object

 mcc                                                                                               description                     _ingested_at                   _source                                                  _content_sha256     _version
0742                                                                                       Veterinary Services 2026-09-25 21:30:50.375080+00:00 github_greggles_mcc_codes 1870af6f01a7b5fa4f89d597f70b3f2481e063e9470549e3a16405f748cb1f9d 9675cfab77d6
0763                                                                                Agricultural Co-operatives 2026-09-25 21:30:50.375080+00: